In [ ]:
#!/usr/bin/env python

import csv
import json
import os
import time
import bisect
from datetime import datetime, timezone

import pandas as pd
import requests
from dateutil import parser
from tqdm import tqdm

# =========================
# CONFIG
# =========================
INPUT_CSV = "all_59k_accidents_raw.csv"

START_ROW = 50000
LIMIT = 5000  # set to None to process to the end

end_marker = START_ROW + LIMIT if LIMIT else "end"
OUTPUT_CSV = f"weather_data_{START_ROW}_to_{end_marker}.csv"
ERRORS_CSV = f"errors_log_{START_ROW}_to_{end_marker}.csv"

RAW_RESP_DIR = "responses_raw"
REQUEST_TIMEOUT = 25
SLEEP_BETWEEN_REQUESTS = 0.0
MAX_RETRIES = 5
BACKOFF_FACTOR = 2.0

ARCHIVE_ENDPOINT = "https://archive-api.open-meteo.com/v1/archive"

# Weather columns only
WEATHER_COLUMNS = [
    "temperature_2m",
    "relativehumidity_2m",
    "dewpoint_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "showers",
    "snowfall",
    "snow_depth",
    "weathercode",
    "pressure_msl",
    "surface_pressure",
    "cloudcover",
    "cloudcover_low",
    "cloudcover_mid",
    "cloudcover_high",
    "windspeed_10m",
    "winddirection_10m",
    "windgusts_10m",
]



os.makedirs(RAW_RESP_DIR, exist_ok=True)


def _parse_datetime_from_row(row):
    raw_date = row.get("accidentdate")
    raw_time = row.get("time")

    if pd.isna(raw_date) or str(raw_date).strip() == "":
        raise ValueError("missing accidentdate")
    if pd.isna(raw_time) or str(raw_time).strip() == "":
        raise ValueError("missing time")

    raw_date = str(raw_date).strip()
    raw_time = str(raw_time).strip()

    try:
        date_part = datetime.strptime(raw_date, "%d-%b-%y")
    except Exception:
        raise ValueError(f"invalid accidentdate format: '{raw_date}'")

    try:
        time_part = datetime.strptime(raw_time, "%H:%M")
    except Exception:
        raise ValueError(f"invalid time format: '{raw_time}'")

    return datetime(
        date_part.year,
        date_part.month,
        date_part.day,
        time_part.hour,
        time_part.minute,
        tzinfo=timezone.utc,
    )
def _request_open_meteo_once(lat, lon, dt_utc):
    date_str = dt_utc.date().isoformat()
    params = {
        "latitude": float(lat),
        "longitude": float(lon),
        "start_date": date_str,
        "end_date": date_str,
        "hourly": ",".join(WEATHER_COLUMNS),
        "timezone": "UTC",
    }
    return requests.get(ARCHIVE_ENDPOINT, params=params, timeout=REQUEST_TIMEOUT)


def request_with_retries(lat, lon, dt_utc):
    attempt = 0
    wait = 2.0

    while attempt < MAX_RETRIES:
        try:
            resp = _request_open_meteo_once(lat, lon, dt_utc)

            if resp.status_code == 429:
                pause_time = 60 if attempt < 2 else 300
                print(f"\n[!] Rate limited (429). Sleeping {pause_time}s before retry {attempt + 1}...")
                time.sleep(pause_time)
                attempt += 1
                continue

            if resp.status_code == 400:
                return False, {
                    "error": f"400 bad request: {resp.text}",
                    "url": resp.url
                }, resp.url

            resp.raise_for_status()
            return True, (resp.json(), resp.url)

        except Exception as e:
            print(f"\n[!] Request error: {e}. Retrying in {wait}s...")
            time.sleep(wait)
            wait *= BACKOFF_FACTOR
            attempt += 1

    return False, "Max retries exceeded"


def _find_best_hour_index(times_list, target_dt_utc):
    target_iso = target_dt_utc.strftime("%Y-%m-%dT%H:%M")

    try:
        idx = times_list.index(target_iso)
        return idx, "exact", times_list[idx]
    except ValueError:
        pos = bisect.bisect_left(times_list, target_iso)
        candidates = [p for p in (pos - 1, pos) if 0 <= p < len(times_list)]

        def iso_to_dt(s):
            return parser.isoparse(s).replace(tzinfo=timezone.utc)

        best = min(
            candidates,
            key=lambda i: abs((iso_to_dt(times_list[i]) - target_dt_utc).total_seconds()),
        )
        return best, "nearest", times_list[best]


def _process_row_sync(idx, row):
    try:
        row = row.fillna("")

        lat = row.get("latitude", row.get("source_latitude"))
        lon = row.get("longitude", row.get("source_longitude"))

        if pd.isna(lat) or pd.isna(lon) or str(lat).strip() == "" or str(lon).strip() == "":
            return idx, False, "missing lat/lon", None

        dt_utc = _parse_datetime_from_row(row)
        if dt_utc is None:
            return idx, False, "missing or unparseable datetime", None

        ok, payload_or_err = request_with_retries(lat, lon, dt_utc)
        if not ok:
            return idx, False, f"request failed: {payload_or_err}", None

        payload, url = payload_or_err

        try:
            with open(os.path.join(RAW_RESP_DIR, f"row_{idx}_raw.json"), "w") as fh:
                json.dump(payload, fh)
        except Exception:
            pass

        hourly = payload.get("hourly", {})
        times = hourly.get("time")
        if not times:
            return idx, False, "no hourly.time in response", url

        idx_hour, match_type, returned_time = _find_best_hour_index(times, dt_utc)

        result_row = {
            "_row_index": idx,
            "source_latitude": lat,
            "source_longitude": lon,
            "weather_latitude": payload.get("latitude"),
            "weather_longitude": payload.get("longitude"),
        }

        for var in WEATHER_COLUMNS:
            series = hourly.get(var)
            result_row[var] = series[idx_hour] if (series and idx_hour < len(series)) else None

        return idx, True, result_row, url

    except Exception as e:
        return idx, False, f"unexpected error: {e}", None


def run_fetcher():
    df = pd.read_csv(INPUT_CSV, dtype=str)
    df.columns = [c.strip() for c in df.columns]
    completed_rows = set()

    if os.path.exists(OUTPUT_JSONL):
        with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    completed_rows.add(obj["_row_index"])
                except Exception:
                    pass

    end_row = START_ROW + LIMIT if LIMIT else len(df)
    df_slice = df.iloc[START_ROW:end_row]

    if os.path.exists(OUTPUT_CSV):
        os.remove(OUTPUT_CSV)
    if os.path.exists(ERRORS_CSV):
        os.remove(ERRORS_CSV)

    output_header_written = False
    errors_header_written = False
    success_count = 0
    error_count = 0

    for i, row in tqdm(df_slice.iterrows(), total=len(df_slice), desc=f"Fetching rows {START_ROW} to {end_row}"):
        idx, ok, payload, url = _process_row_sync(i, row)

        if ok:
            out_df = pd.DataFrame([payload], columns=OUTPUT_COLUMNS)
            out_df.to_csv(OUTPUT_CSV, index=False, mode="a", header=not output_header_written)
            output_header_written = True
            success_count += 1
        else:
            with open(ERRORS_CSV, "a", newline="") as ef:
                writer = csv.writer(ef)
                if not errors_header_written:
                    writer.writerow(["row_index", "error", "url"])
                    errors_header_written = True
                writer.writerow([idx, payload, url])
            error_count += 1

        if SLEEP_BETWEEN_REQUESTS:
            time.sleep(SLEEP_BETWEEN_REQUESTS)

    print(f"\nDone. Saved to {OUTPUT_CSV}. Successes: {success_count}, Errors: {error_count}")


if __name__ == "__main__":
    if os.path.exists(INPUT_CSV):
        completed_rows = set()

        if os.path.exists(OUTPUT_JSONL):
            with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
                for line in f:
                    try:
                        obj = json.loads(line)
                        completed_rows.add(obj["_row_index"])
                    except Exception:
                        pass

        else:
            print(f"Error: {INPUT_CSV} not found in current directory.")

In [ ]:
# Getting archive and forecast

#!/usr/bin/env python3

import csv
import json
import os
import time
from datetime import datetime, timezone

import pandas as pd
import requests
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================
INPUT_CSV = "all_59k_accidents_raw.csv"
OUTPUT_JSONL = "open_meteo_archive_and_forecast_full.jsonl"
ERRORS_CSV = "open_meteo_errors.csv"

# RAW_RESP_DIR = "responses_raw"
# ARCHIVE_RAW_DIR = os.path.join(RAW_RESP_DIR, "archive")
# FORECAST_RAW_DIR = os.path.join(RAW_RESP_DIR, "forecast")

# os.makedirs(ARCHIVE_RAW_DIR, exist_ok=True)
# os.makedirs(FORECAST_RAW_DIR, exist_ok=True)

REQUEST_TIMEOUT = 30
MAX_RETRIES = 5
BACKOFF_FACTOR = 2.0
SLEEP_BETWEEN_REQUESTS = 0.0

# Hard stop for the run, so you stay under your monthly cap.
MONTHLY_CALL_BUDGET = int(os.getenv("OPEN_METEO_CALL_BUDGET", "5000000"))

# Set these via env if needed.
OPEN_METEO_API_KEY = os.getenv("OPEN_METEO_API_KEY", "").strip()
ARCHIVE_ENDPOINT = os.getenv(
    "OPEN_METEO_ARCHIVE_ENDPOINT",
    "https://archive-api.open-meteo.com/v1/archive",
)
FORECAST_ENDPOINT = os.getenv(
    "OPEN_METEO_FORECAST_ENDPOINT",
    "https://customer-api.open-meteo.com/v1/forecast",
)

# Open-Meteo controls
TIMEZONE = "UTC"
TIMEFORMAT = "iso8601"
CELL_SELECTION = "land"

# Use the provider’s combined default unless you want to override it.
ARCHIVE_MODELS = os.getenv("OPEN_METEO_ARCHIVE_MODELS", "best_match")
FORECAST_MODELS = os.getenv("OPEN_METEO_FORECAST_MODELS", "best_match")

# If set, these override Open-Meteo’s default downscaling behavior.
# Leave blank to omit the parameter.
ELEVATION_OVERRIDE = os.getenv("OPEN_METEO_ELEVATION_OVERRIDE", "").strip()

# Batch size per request. Keeping it at 10 is conservative.
BATCH_SIZE = 10

session = requests.Session()
calls_made = 0


# ============================================================
# VARIABLE SETS
# ============================================================

# Archive hourly variables (verified from the current archive docs)
ARCHIVE_HOURLY_VARS = [
    "temperature_2m",
    "relativehumidity_2m",
    "dewpoint_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "snow_depth",
    "weathercode",
    "pressure_msl",
    "surface_pressure",
    "cloudcover",
    "cloudcover_low",
    "cloudcover_mid",
    "cloudcover_high",
    "windspeed_10m",
    "winddirection_10m",
    "windgusts_10m",
    "et0_fao_evapotranspiration",
    "vapour_pressure_deficit",
    "windspeed_100m",
    "winddirection_100m",
    "soil_temperature_0_to_7cm",
    "soil_temperature_7_to_28cm",
    "soil_temperature_28_to_100cm",
    "soil_temperature_100_to_255cm",
    "soil_moisture_0_to_7cm",
    "soil_moisture_7_to_28cm",
    "soil_moisture_28_to_100cm",
    "soil_moisture_100_to_255cm",
]

ARCHIVE_DAILY_VARS = [
    "weathercode",
    "temperature_2m_max",
    "temperature_2m_min",
    "apparent_temperature_max",
    "apparent_temperature_min",
    "sunrise",
    "sunset",
    "daylight_duration",
    "sunshine_duration",
    "precipitation_sum",
    "rain_sum",
    "snowfall_sum",
    "precipitation_hours",
    "windspeed_10m_max",
    "windgusts_10m_max",
    "winddirection_10m_dominant",
    "shortwave_radiation_sum",
    "et0_fao_evapotranspiration",
]

# Forecast current variables (verified from the current forecast docs)
FORECAST_CURRENT_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "apparent_temperature",
    "is_day",
    "precipitation",
    "rain",
    "showers",
    "snowfall",
    "weather_code"
]

# Forecast hourly variables (verified from the current forecast docs)
FORECAST_HOURLY_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "apparent_temperature",
    "precipitation_probability",
    "precipitation",
    "rain",
    "showers",
    "snowfall",
    "snow_depth",
    "weather_code",
    "cloud_cover",
    "cloud_cover_low",
    "cloud_cover_mid",
    "cloud_cover_high",
    "visibility",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
]

# Forecast 15-min variables (verified from the current forecast docs)
FORECAST_MINUTELY_15_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "weather_code"
]

# Pressure-level variables are hourly variables in the forecast API.
PRESSURE_LEVELS = [
    1000, 975, 950, 925, 900, 850, 800, 700, 600, 500,
    400, 300, 250, 200, 150, 100, 70, 50, 30,
]
PRESSURE_LEVEL_FAMILIES = [
    "temperature",
    "relative_humidity",
    "dew_point",
    "cloudcover",
    "wind_speed",
    "wind_direction",
    "geopotential_height",
]
FORECAST_PRESSURE_LEVEL_VARS = [
    f"{family}_{level}hPa"
    for family in PRESSURE_LEVEL_FAMILIES
    for level in PRESSURE_LEVELS
]
FORECAST_HOURLY_VARS += FORECAST_PRESSURE_LEVEL_VARS

# CSV columns
OUTPUT_COLUMNS = [
    "_row_index",
    "accident_datetime_utc",
    "source_latitude",
    "source_longitude",
    "archive_status",
    "archive_error",
    "archive_request_count",
    "archive_json",
    "forecast_status",
    "forecast_error",
    "forecast_request_count",
    "forecast_json",
]


# ============================================================
# HELPERS
# ============================================================

def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def parse_accident_datetime_utc(row):
    raw_date = str(row.get("accidentdate", "")).strip()
    raw_time = str(row.get("time", "")).strip()

    if not raw_date:
        raise ValueError("missing accidentdate")
    if not raw_time:
        raise ValueError("missing time")

    try:
        date_part = datetime.strptime(raw_date, "%d-%b-%y")
    except Exception as exc:
        raise ValueError(f"invalid accidentdate format: {raw_date}") from exc

    try:
        time_part = datetime.strptime(raw_time, "%H:%M")
    except Exception as exc:
        raise ValueError(f"invalid time format: {raw_time}") from exc

    return datetime(
        date_part.year,
        date_part.month,
        date_part.day,
        time_part.hour,
        time_part.minute,
        tzinfo=timezone.utc,
    )




def get_base_params(lat, lon, extra_params=None):
    params = {
        "latitude": float(lat),
        "longitude": float(lon),
        "timezone": TIMEZONE,
        "timeformat": TIMEFORMAT,
        "cell_selection": CELL_SELECTION,
        "models": ARCHIVE_MODELS if extra_params and extra_params.get("_endpoint") == "archive" else FORECAST_MODELS,
    }

    if ELEVATION_OVERRIDE != "":
        params["elevation"] = ELEVATION_OVERRIDE

    if OPEN_METEO_API_KEY:
        params["apikey"] = OPEN_METEO_API_KEY

    if extra_params:
        for k, v in extra_params.items():
            if k != "_endpoint" and v is not None:
                params[k] = v

    return params


def merge_payloads(target, source):
    if not target:
        target = {}

    for key, value in source.items():
        if key in {"hourly", "hourly_units", "daily", "daily_units", "current", "current_units", "minutely_15", "minutely_15_units"}:
            if key not in target:
                target[key] = {}
            if isinstance(value, dict):
                target[key].update(value)
        else:
            if key not in target:
                target[key] = value
    return target


def request_json(endpoint, params):
    global calls_made

    attempt = 0
    wait = 2.0
    last_url = None

    while attempt < MAX_RETRIES:
        try:
            calls_made += 1
            resp = session.get(endpoint, params=params, timeout=REQUEST_TIMEOUT)
            last_url = resp.url

            if resp.status_code == 429:
                pause = 60 if attempt < 2 else 300
                print(f"\n[!] Rate limited (429). Sleeping {pause}s before retry {attempt + 1}...")
                time.sleep(pause)
                attempt += 1
                continue

            if resp.status_code == 400:
                return False, {
                    "error": f"400 bad request: {resp.text}",
                    "url": resp.url
                }, resp.url

            resp.raise_for_status()
            return True, resp.json(), resp.url

        except Exception as exc:
            print(f"\n[!] Request error: {exc}. Retrying in {wait}s...")
            time.sleep(wait)
            wait *= BACKOFF_FACTOR
            attempt += 1

    return False, {"error": "Max retries exceeded", "url": last_url}, last_url


def fetch_batched_section(endpoint, section_key, variables, base_params, batch_prefix):
    merged = {}
    urls = []
    request_count = 0

    for batch_idx, batch in enumerate(chunked(variables, BATCH_SIZE), start=1):
        params = dict(base_params)
        params[section_key] = ",".join(batch)

        ok, payload_or_err, url = request_json(endpoint, params)
        request_count += 1

        if not ok:
            return (
                False,
                merged,
                urls,
                request_count,
                payload_or_err.get("error", "request failed"),
            )

        payload = payload_or_err
        merged = merge_payloads(merged, payload)
        urls.append(url)

    return True, merged, urls, request_count, ""


def fetch_archive_for_row(row_idx, lat, lon, accident_dt):
    base_params = get_base_params(lat, lon, {"_endpoint": "archive"})
    base_params.update({
        "start_date": accident_dt.date().isoformat(),
        "end_date": accident_dt.date().isoformat(),
        "hourly": None,
        "daily": None,
    })

    archive = {}
    total_requests = 0
    errors = []

    ok_h, hourly_payload, hourly_urls, reqs, err = fetch_batched_section(
        ARCHIVE_ENDPOINT,
        "hourly",
        ARCHIVE_HOURLY_VARS,
        base_params,
        "hourly",
    )
    total_requests += reqs
    if ok_h:
        archive = merge_payloads(archive, hourly_payload)
    else:
        errors.append(f"hourly: {err}")

    ok_d, daily_payload, daily_urls, reqs, err = fetch_batched_section(
        ARCHIVE_ENDPOINT,
        "daily",
        ARCHIVE_DAILY_VARS,
        base_params,
        "daily",
    )
    total_requests += reqs
    if ok_d:
        archive = merge_payloads(archive, daily_payload)
    else:
        errors.append(f"daily: {err}")

    archive_meta = {
        "endpoint": ARCHIVE_ENDPOINT,
        "urls": hourly_urls + daily_urls,
        "requests": total_requests,
        "errors": errors,
    }
    return len(errors) == 0, archive_meta, archive


def fetch_forecast_for_row(row_idx, lat, lon):
    base_params = get_base_params(lat, lon, {"_endpoint": "forecast"})
    base_params.update({
        "forecast_days": 16,
        "past_days": 0,
        "hourly": None,
        "daily": None,
        "current": None,
        "minutely_15": None,
    })

    forecast = {}
    total_requests = 0
    errors = []

    ok_c, current_payload, current_urls, reqs, err = fetch_batched_section(
        FORECAST_ENDPOINT,
        "current",
        FORECAST_CURRENT_VARS,
        base_params,
        "current",
    )
    total_requests += reqs
    if ok_c:
        forecast = merge_payloads(forecast, current_payload)
    else:
        errors.append(f"current: {err}")

    ok_h, hourly_payload, hourly_urls, reqs, err = fetch_batched_section(
        FORECAST_ENDPOINT,
        "hourly",
        FORECAST_HOURLY_VARS,
        base_params,
        "hourly",
    )
    total_requests += reqs
    if ok_h:
        forecast = merge_payloads(forecast, hourly_payload)
    else:
        errors.append(f"hourly: {err}")

    ok_m, minutely_payload, minutely_urls, reqs, err = fetch_batched_section(
        FORECAST_ENDPOINT,
        "minutely_15",
        FORECAST_MINUTELY_15_VARS,
        base_params,
        "minutely15",
    )
    total_requests += reqs
    if ok_m:
        forecast = merge_payloads(forecast, minutely_payload)
    else:
        errors.append(f"minutely_15: {err}")

    ok_d, daily_payload, daily_urls, reqs, err = fetch_batched_section(
        FORECAST_ENDPOINT,
        "daily",
        [
            "weathercode",
            "temperature_2m_max",
            "temperature_2m_min",
            "apparent_temperature_max",
            "apparent_temperature_min",
            "sunrise",
            "sunset",
            "daylight_duration",
            "sunshine_duration",
            "uv_index_max",
            "uv_index_clear_sky_max",
            "precipitation_sum",
            "rain_sum",
            "showers_sum",
            "snowfall_sum",
            "precipitation_hours",
            "precipitation_probability_max",
            "precipitation_probability_mean",
            "precipitation_probability_min",
            "windspeed_10m_max",
            "windgusts_10m_max",
            "winddirection_10m_dominant",
            "shortwave_radiation_sum",
            "et0_fao_evapotranspiration",
        ],
        base_params,
        "daily",
    )
    total_requests += reqs
    if ok_d:
        forecast = merge_payloads(forecast, daily_payload)
    else:
        errors.append(f"daily: {err}")

    forecast_meta = {
        "endpoint": FORECAST_ENDPOINT,
        "urls": current_urls + hourly_urls + minutely_urls + daily_urls,
        "requests": total_requests,
        "errors": errors,
        "reference_time_utc": datetime.now(timezone.utc).isoformat(),
    }
    return len(errors) == 0, forecast_meta, forecast

def empty_payload():
    return {}


# ============================================================
# MAIN LOOP
# ============================================================

def run_fetcher():
    df = pd.read_csv(INPUT_CSV, dtype=str)
    df.columns = [c.strip() for c in df.columns]
    if not os.path.exists(OUTPUT_JSONL):
        open(OUTPUT_JSONL, "a").close()
    if os.path.exists(ERRORS_CSV):
        os.remove(ERRORS_CSV)

    errors_header_written = False

    success_rows = 0
    error_rows = 0

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Fetching accident rows"):
        if idx in completed_rows:
            continue
        if calls_made >= MONTHLY_CALL_BUDGET:
            print(
                f"\n[!] Monthly call budget reached "
                f"({calls_made} >= {MONTHLY_CALL_BUDGET}). Stopping."
            )
            break

        try:
            row = row.fillna("")
            lat_raw = row.get("latitude", row.get("source_latitude"))
            lon_raw = row.get("longitude", row.get("source_longitude"))

            if pd.isna(lat_raw) or pd.isna(lon_raw) or str(lat_raw).strip() == "" or str(lon_raw).strip() == "":
                raise ValueError("missing lat/lon")

            lat = float(str(lat_raw).strip())
            lon = float(str(lon_raw).strip())
            accident_dt = parse_accident_datetime_utc(row)

            archive_ok, archive_meta, archive_payload = fetch_archive_for_row(
                idx, lat, lon, accident_dt
            )

            forecast_ok, forecast_meta, forecast_payload = fetch_forecast_for_row(
                idx, lat, lon
            )


            out_row = {
                "_row_index": idx,
                "accident_datetime_utc": accident_dt.isoformat(),
                "source_latitude": lat,
                "source_longitude": lon,

                "archive_status": "ok" if archive_ok else "partial",
                "archive_error": archive_meta["errors"],
                "archive_request_count": archive_meta["requests"],
                "archive": archive_payload,

                "forecast_status": "ok" if forecast_ok else "partial",
                "forecast_error": forecast_meta["errors"],
                "forecast_request_count": forecast_meta["requests"],
                "forecast": forecast_payload,
            }

            with open(OUTPUT_JSONL, "a", encoding="utf-8") as f:
                f.write(json.dumps(out_row, ensure_ascii=False) + "\n")
            success_rows += 1

            if not archive_ok or not forecast_ok:
                with open(ERRORS_CSV, "a", newline="", encoding="utf-8") as ef:
                    writer = csv.writer(ef)
                    if not errors_header_written:
                        writer.writerow(["row_index", "type", "message"])
                        errors_header_written = True
                    writer.writerow([
                        idx,
                        "partial",
                        f"archive={archive_meta['errors']} forecast={forecast_meta['errors']}",
                    ])

        except Exception as exc:
            error_rows += 1
            with open(ERRORS_CSV, "a", newline="", encoding="utf-8") as ef:
                writer = csv.writer(ef)
                if not errors_header_written:
                    writer.writerow(["row_index", "type", "message"])
                    errors_header_written = True
                writer.writerow([idx, "fatal", str(exc)])

        if SLEEP_BETWEEN_REQUESTS:
            time.sleep(SLEEP_BETWEEN_REQUESTS)

    print(
        f"\nDone. Rows saved: {success_rows}, errors: {error_rows}, "
        f"HTTP calls made: {calls_made}"
    )


if __name__ == "__main__":
    if not os.path.exists(INPUT_CSV):
        print(f"Error: {INPUT_CSV} not found in current directory.")
        raise SystemExit(1)

    run_fetcher()